# MVSep MDX23 Colab v2.1.1
* Slow separations fixed, unified installation cells (28.12.23).
* Chunks currently set to 100000, work with 3:00 track. Decrease it if you run out of memory.
* With current settings it takes 35 minutes to separate 3:00 track. For faster separation set both overlaps to 0.8, as it's still good performance/quality parameter (0.96 max tested, with 300K before memory issues).
<br><br>
* Don't write file names in the input field below - only folder paths allowed (otherwise it will show just /content/MVSEP-MDX23-Colab_v2.1 error).
* Batch processing - all audio files inside provided input folder will be separated.<br>
* Not all EOFError errors are critical. Check your output folder after separation first.<br>
* [v2.2 final](https://colab.research.google.com/github/jarredou/MVSEP-MDX23-Colab_v2/blob/v2.2/MVSep-MDX23-Colab.ipynb) (with faster optional only 2 stem output) - better SDR, more vocal residues

Colab version of MDX23 algorithm from [MVSEP.COM](https://www.mvsep.com) with some tweaks:
* Updated with new UVR-MDX': voc_ft & Instr-HQ3 models
* Fixed high frequency bleed in vocals
* Fixed volume compensation for MDX models
* It receives input files from temp Colab disk (open file manager in the left to upload a file), not from GDrive, but you can change it to GDrive by setting /content/drive/MyDrive/input (be aware that the path in some cases is case-sensitive and you must create that folder on your own)
* For a bit better quality, chunks_size can be set to 500K for ~2:41-3:58 tracks - 500K fails with 5 minute tracks
* Replace inference.py by [that](https://raw.githubusercontent.com/Infisrael/MVSEP-MDX23-Colab_v2.1/main/inference.py) one if you still have memory issues with 5 minute tracks and chunk_size set to 300K (open file manager and go to Colab folder) - be aware that it disables Demucs denoiser and increases vocal residues.
* Beta 2.2 1.5.1 [inference](https://cdn.discordapp.com/attachments/887455924845944873/1129470812580225164/inference_1.5.1_vocft_mod.py) - might be a bit cleaner (denoiser still on).
* 2.2 Pre Beta 3 /wo v3 MDX yet [Colab](https://colab.research.google.com/github/jarredou/MVSEP-MDX23-Colab_v2/blob/597b5b7f653e4593a0a94938a3923077d66f8767/MVSep-MDX23-Colab.ipynb) and [inference](https://drive.google.com/file/d/1bpZKZynmdsYcriF-M8t8yLtVLm7zRz5U/view?usp=sharing) - might be even cleaner (denoiser still on) - delete two bigshifts and vocals only references in the cell to make it work.
* Beta 2.2 1.5.2 (shifts 0, less muddy) - [inference](https://drive.google.com/file/d/1GKTShwgVKwyLNLssh2AmLiN5vEwrUPus/view?usp=sharing).
* For both overlaps set to 0.95 and 500K chunks it takes 27 minutes to process 3:58 track with shifts 0 set in inference from above.
* It will show a lot of warinings and exceptions (e.g. PySoundFile failed, EOFError) frequently, but separation can be successful with these errors (and unsuccessful too, if you set wrong parameters)
* Instrum is inverted vocals stem
* Instrum2 is the sum of drums+bass+other stems (muddier, smaller SDR)
* It doesn't use Demucs denoiser disabled yet, so it's less noisy here than current 2.1 - that fix in 2.1 was introduced after the 2.1 release in 2.0 repo (fixes problems when no matter what chunks you use, you still get memory errors in e.g. Demucs 3/4 step)


Credits:
* [https://github.com/ZFTurbo/MVSEP-MDX23-music-separation-model](https://github.com/ZFTurbo/MVSEP-MDX23-music-separation-model)
* Models by [Demucs](https://github.com/facebookresearch/demucs), [UVR GUI Team](https://github.com/Anjok07/ultimatevocalremovergui) - Anjok / Aufr33 & [Kimberley Jensen](https://github.com/KimberleyJensen)
* Adaptation & tweaks by [jarredou](https://github.com/jarredou/MVSEP-MDX23-Colab_v2/)

In [2]:
#@markdown #Installation
#@markdown *Run this cell to install MVSep-MDX23*
print('Installing... This will take 1 minute...')
%cd /content
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/deton24/MVSEP-MDX23-Colab_v2.1 &> /dev/null
%cd /content/MVSEP-MDX23-Colab_v2.1
!pip install -r requirements.txt &> /dev/null
!pip install demucs
!pip install onnxruntime-gpu
# onnxruntime-gpu nightly fix for cuda12.2
!python -m pip install ort-nightly-gpu --index-url=https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/ort-cuda-12-nightly/pypi/simple/
print('Installation done !')

Installing... This will take 1 minute...
/content
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/MVSEP-MDX23-Colab_v2.1
Looking in indexes: https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/ort-cuda-12-nightly/pypi/simple/
Installation done !


In [42]:
import os

inference_file_path = '/content/MVSEP-MDX23-Colab_v2.1/inference.py'

try:
    # 1. First, try to find a backup or re-read the original file state
    # If the file is too small, it means we accidentally wiped it.
    # In that case, we need to reset it from the repo.
    if os.path.getsize(inference_file_path) < 1000:
        print("Restoring inference.py from repository...")
        !cd /content/MVSEP-MDX23-Colab_v2.1 && git checkout inference.py

    with open(inference_file_path, 'r') as f:
        lines = f.readlines()

    # 2. Identify the start of the actual logic (usually __author__ or similar)
    # We skip all lines that look like our previous patching attempts
    patch_keywords = ["nmport", "add_safe_globals", "HTDemucs", "Float64DType", "fractions", "# coding: utf-8"]
    original_logic_start = 0
    for i, line in enumerate(lines):
        if "__author__" in line or "import argparse" in line or "import time" in line:
            original_logic_start = i
            break

    original_code = lines[original_logic_start:]

    # 3. Create the clean, functional header
    header = [
        "# coding: utf-8\n",
        "import torch\n",
        "import torch.nn as nn\n",
        "import demucs.htdemucs\n",
        "import fractions\n",
        "import numpy\n",
        "import numpy as np\n",
        "try:\n",
        "    torch.serialization.add_safe_globals([demucs.htdemucs.HTDemucs, fractions.Fraction, numpy.core.multiarray.scalar, numpy.dtype, numpy.dtypes.Float64DType])\n",
        "except (ImportError, AttributeError):\n",
        "    pass\n",
        "\n"
    ]

    final_content = "".join(header) + "".join(original_code)

    with open(inference_file_path, 'w') as f:
        f.write(final_content)

    print("Successfully restored and patched inference.py. Please re-run the separation cell.")

except Exception as e:
    print(f"An error occurred during restoration: {e}")

Restoring inference.py from repository...
Updated 1 path from the index
Successfully restored and patched inference.py. Please re-run the separation cell.


In [43]:
import os
from pathlib import Path
import glob

%cd /content/MVSEP-MDX23-Colab_v2.1
def console(t):
    get_ipython().system(t)

#file_path = '/content/drive/MyDrive/mvsep_dataset/' #@param {type:"string"}
input_path = '/content/Smoking Behind the Supermarket with You. Mini Episodes Episode 1.mp3' #@param {type:"string"}
output_folder = '/content/drive/MyDrive/output' #@param {type:"string"}
overlap_large = 0.9 #@param {type:"slider", min:0.1, max:1, step:0.05}
overlap_small = 0.9 #@param {type:"slider", min:0.1, max:1, step:0.05}

#@markdown *Use lower chunk_size if you have memory errors*
chunk_size = 100000 #@param {type:"slider", min:50000, max:1000000, step:100000}

already_separated_files = []

if os.path.isfile(input_path):
    # If input_path is a file, process it directly
    file_paths_to_process = [input_path]
elif os.path.isdir(input_path):
    # If input_path is a directory, find all audio files within it
    file_paths_to_process = glob.glob(os.path.join(input_path, "*"))
else:
    print(f"Error: '{input_path}' is neither a file nor a directory.")
    file_paths_to_process = []

for file_path in file_paths_to_process:
  filename =  Path(file_path).stem
  specific_output_folder = Path(output_folder, filename)

  if filename in already_separated_files: #if file is already processed skip
    print(f"Skipping {filename} as it's already processed.")
    continue

  specific_output_folder.mkdir(parents=True, exist_ok=True)
  console(f'python inference.py --large_gpu --chunk_size {chunk_size} --input_audio "{file_path}" --overlap_large {overlap_large} --overlap_small {overlap_small} --output_folder "{specific_output_folder}"')

/content/MVSEP-MDX23-Colab_v2.1
GPU use: 0
Options: 
input_audio: ['/content/Smoking Behind the Supermarket with You. Mini Episodes Episode 1.mp3']
output_folder: /content/drive/MyDrive/output/Smoking Behind the Supermarket with You. Mini Episodes Episode 1
cpu: False
overlap_large: 0.9
overlap_small: 0.9
single_onnx: False
chunk_size: 100000
large_gpu: True
Use fast large GPU memory version of code
Use device: cuda:0
Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/f7e0c4bc-ba3fe64a.th" to /root/.cache/torch/hub/checkpoints/f7e0c4bc-ba3fe64a.th
100% 80.2M/80.2M [00:00<00:00, 140MB/s]
Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/d12395a8-e57c48e6.th" to /root/.cache/torch/hub/checkpoints/d12395a8-e57c48e6.th
100% 80.2M/80.2M [00:00<00:00, 101MB/s]
Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/92cfc3b6-ef3bcb9c.th" to /root/.cache/torch/hub/checkpoints/92cfc3b6-ef3bcb9c.th
100% 80.2M/80.2M [00:00<00:00, 197MB/s]
Do

In [ ]:
import os

# 1. Fix missing libcudnn for ONNX/GPU acceleration
print("Fixing GPU library links...")
!ln -sf /usr/lib/x86_64-linux-gnu/libcudnn.so.8 /usr/lib/x86_64-linux-gnu/libcudnn.so.9

# 2. Optimized Separation (Faster Settings)
input_path = '/content/Smoking Behind the Supermarket with You. Mini Episodes Episode 1.mp3'
output_folder = '/content/drive/MyDrive/output'

# Reducing overlap to 0.6 significantly speeds up processing without major quality loss
overlap_large = 0.6
overlap_small = 0.6
chunk_size = 300000 # Increased chunk size for better GPU utilization

from pathlib import Path
import glob

%cd /content/MVSEP-MDX23-Colab_v2.1

file_paths = [input_path] if os.path.isfile(input_path) else glob.glob(os.path.join(input_path, "*"))

for file_path in file_paths:
    filename = Path(file_path).stem
    specific_output_folder = Path(output_folder, filename)
    specific_output_folder.mkdir(parents=True, exist_ok=True)

    print(f"--- Starting Optimized Separation for: {filename} ---")
    !python inference.py --large_gpu --chunk_size {chunk_size} --input_audio "{file_path}" --overlap_large {overlap_large} --overlap_small {overlap_small} --output_folder "{specific_output_folder}"


Fixing GPU library links...
/content/MVSEP-MDX23-Colab_v2.1
--- Starting Optimized Separation for: Smoking Behind the Supermarket with You. Mini Episodes Episode 1 ---
GPU use: 0
Options: 
input_audio: ['/content/Smoking Behind the Supermarket with You. Mini Episodes Episode 1.mp3']
output_folder: /content/drive/MyDrive/output/Smoking Behind the Supermarket with You. Mini Episodes Episode 1
cpu: False
overlap_large: 0.6
overlap_small: 0.6
single_onnx: False
chunk_size: 300000
large_gpu: True
Use fast large GPU memory version of code
Use device: cuda:0
Model path: /content/MVSEP-MDX23-Colab_v2.1/models/UVR-MDX-NET-Voc_FT.onnx
Device: cuda:0 Chunk size: 300000
2026-06-14 14:00:03.504457561 [E:onnxruntime:Default, provider_bridge_ort.cc:1836 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1511 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with erro

In [ ]:
#@title (optional) Mixdown to 64 bit (might be slightly better instrum2 equivalent (32-bit float) - can be placebo in most cases; (change it to pcm_s16le for 16 bit). Write file name without input file extension in paths below

!ffmpeg -i "/content/drive/MyDrive/output/your track/your track_bass.wav" -i "/content/drive/MyDrive/output/your track/your track_other.wav" -i "/content/drive/MyDrive/output/your track/your track_drums.wav" -filter_complex "[0]volume=3[a];[1]volume=3[b];[2]volume=3[c];[a][b][c]amix=inputs=3:duration=longest" -c:a pcm_f64le '/content/drive/MyDrive/output/output.wav'